In [16]:
from pinecone import Pinecone, ServerlessSpec
import os
from openai import OpenAI
import pandas as pd
from time import time
import dotenv
dotenv.load_dotenv()


from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.document_loaders import DirectoryLoader, PyPDFLoader, UnstructuredPowerPointLoader


In [2]:
token= os.getenv("RUNPOD_TOKEN") 
open_ai_base_url = os.getenv("RUNPOD_EMBEDDING_URL") 
model_name= os.getenv("MODEL_NAME") 
pinecone_api_key = os.getenv("PINECONE_API_KEY") 

In [3]:
pc = Pinecone(api_key=pinecone_api_key)

client = OpenAI(
  api_key=token, 
  base_url=open_ai_base_url
)

## Try out embeddings

In [4]:
output = client.embeddings.create(input = ["helloo there"],model=model_name)
embedings = output.data[0].embedding
print(embedings)

[-0.05535165220499039, -0.056572869420051575, 0.08585154265165329, -0.06237364932894707, 0.01749393157660961, -0.011418377049267292, 0.052420731633901596, 0.052603915333747864, 0.02895810268819332, -0.02263830602169037, -0.013250201940536499, -0.04787169769406319, 0.029446590691804886, 0.031751636415719986, 0.05635915696620941, -0.009006474167108536, 0.012746450491249561, -0.054222024977207184, -0.103620246052742, -0.019554734230041504, 0.029156550765037537, 0.05199330672621727, -0.028805451467633247, -0.034804679453372955, 0.0015770869795233011, -0.006640366278588772, 0.019661590456962585, 0.03562900051474571, -0.008487456478178501, -0.07376149296760559, -0.00048514746595174074, -0.012944898568093777, 0.04658942297101021, -0.0003446598129812628, 0.051382698118686676, 0.0005123385926708579, 0.059625908732414246, -0.025065474212169647, -0.07791363447904587, -0.005609964486211538, 0.0604502335190773, -0.025019679218530655, -0.0016944382805377245, -0.016669608652591705, 0.0225619804114103

In [5]:
len(embedings)

384

## Wrangle dataset

In [6]:
df=pd.read_json('data/students.json')

In [7]:
df.head(2)

,full_name,registration_number,email,region,description,image_path,year,department,status
0,Hiba Mansour,S12255237,s12255237@stu.najah.edu,Palestine,Computer science student with a passion for da...,hiba_mansour.jpg,3rd Year,Computer Science,active
1,Mai Shelbayeh,N/A,Maishelbayeh@icloud.com,Palestine,AI enthusiast with interest in mobile app deve...,mai_shelbayeh.jpg,1st Year,Software Engineering,active


In [8]:
df['text'] =  df['full_name']+" : "+df['description'] + \
                " -- Registration Number: " + df['registration_number'] + \
                " -- Email: " + df['email'] + \
                " -- Department: " + df['department'] + \
                " -- Year: " + df['year'] + \
                " -- Status: " + df['status']

In [9]:
# Show full text without truncation
pd.set_option('display.max_colwidth', None)

# Show the first row's text column
print(df['text'].head(1))

0    Hiba Mansour : Computer science student with a passion for data analysis. -- Registration Number: S12255237 -- Email: s12255237@stu.najah.edu -- Department: Computer Science -- Year: 3rd Year -- Status: active
Name: text, dtype: object


In [10]:
df['text'].head()

0          Hiba Mansour : Computer science student with a passion for data analysis. -- Registration Number: S12255237 -- Email: s12255237@stu.najah.edu -- Department: Computer Science -- Year: 3rd Year -- Status: active
1               Mai Shelbayeh : AI enthusiast with interest in mobile app development. -- Registration Number: N/A -- Email: Maishelbayeh@icloud.com -- Department: Software Engineering -- Year: 1st Year -- Status: active
2    Orwa Jabali : Interested in smart systems and human-computer interaction. -- Registration Number: s12255138 -- Email: Orwajabali89@gmail.com -- Department: Artificial Intelligence -- Year: 3rd Year -- Status: active
3              Raghad Hethnawi : Aspiring researcher in natural language processing. -- Registration Number: S12457356 -- Email: s12457356@stu.najah.edu -- Department: Computer Science -- Year: 1st Year -- Status: active
4                     Aseel Omar : Focused on cybersecurity and network systems. -- Registration Number: S12356791 -

In [11]:
texts = df['text'].tolist()

In [12]:
with open('data/Prof_Allam_Mousa_CV.txt') as f:
    Prof_Allam_Mousa_CV = f.read()
    
Prof_Allam_Mousa_CV = "Professor Allam Mousa - Curriculum Vitae: " + Prof_Allam_Mousa_CV
texts.append(Prof_Allam_Mousa_CV)

In [13]:
with open('data/SSM_Syllabus.txt') as f:
    SSM_Syllabus = f.read()
    
SSM_Syllabus = "Course Syllabus: " + SSM_Syllabus
texts.append(SSM_Syllabus)

In [14]:
texts

['Hiba Mansour : Computer science student with a passion for data analysis. -- Registration Number: S12255237 -- Email: s12255237@stu.najah.edu -- Department: Computer Science -- Year: 3rd Year -- Status: active',
 'Mai Shelbayeh : AI enthusiast with interest in mobile app development. -- Registration Number: N/A -- Email: Maishelbayeh@icloud.com -- Department: Software Engineering -- Year: 1st Year -- Status: active',
 'Orwa Jabali : Interested in smart systems and human-computer interaction. -- Registration Number: s12255138 -- Email: Orwajabali89@gmail.com -- Department: Artificial Intelligence -- Year: 3rd Year -- Status: active',
 'Raghad Hethnawi : Aspiring researcher in natural language processing. -- Registration Number: S12457356 -- Email: s12457356@stu.najah.edu -- Department: Computer Science -- Year: 1st Year -- Status: active',
 'Aseel Omar : Focused on cybersecurity and network systems. -- Registration Number: S12356791 -- Email: s12356791@stu.najah.edu -- Department: Inf

In [17]:

## Split Data into Text Chunks
def text_split(extracted_data):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=20
    )
    text_chunks = text_splitter.split_documents(extracted_data)
    return text_chunks

In [18]:
def load_documents(directory_path):
    # Load PDF files
    pdf_loader = DirectoryLoader(
        path=directory_path,
        glob="*.pdf",
        loader_cls=PyPDFLoader
    )
    pdf_docs = pdf_loader.load()

    # Load PPTX files
    pptx_loader = DirectoryLoader(
        path=directory_path,
        glob="*.pptx",
        loader_cls=UnstructuredPowerPointLoader
    )
    pptx_docs = pptx_loader.load()

    # Combine both
    all_docs = pdf_docs + pptx_docs
    return all_docs


In [19]:
docs = load_documents("data/books_slides")
print(f"Loaded {len(docs)} documents.")


Loaded 4544 documents.


In [78]:
# Split Data into Text Chunks
text_chunks = text_split(docs)

In [79]:
text_chunks[1]

Document(metadata={'producer': 'Adobe Acrobat 8.0', 'creator': 'Adobe Acrobat 8.0 Combine Files', 'creationdate': '2008-03-06T00:59:51+04:00', 'author': 'Roger S Pressman', 'moddate': '2010-07-10T12:37:12-04:00', 'title': "Software Engineering: A Practitioner's Approach", 'ebx_publisher': 'McGraw-Hill Higher Education', 'source': 'data\\books_slides\\16_EBOOK-7th_ed_software_engineering_a_practitioners_approach_by_roger_s._pressman_.pdf', 'total_pages': 930, 'page': 0, 'page_label': 'C'}, page_content='engineering and where is it now?’ ACM Computing Reviews\n“An up-to-the minute, in-depth treatment of the software engineering process.”\nByte Book Club (main selection)\n“... had the best explanations of what I want to cover ...”\n“... The de/f_i  nitive book on the subject as far as I’m concerned ...”\n“... A good textbook as well as reference ...” from comp.software-eng FAQ\n“As a practicing Software Engineer, I /f_i  nd this book to be invaluable. It has served as')

In [80]:
# Combine all texts
combined_text = texts + text_chunks
print(f"Total number of text chunks to embed: {len(combined_text)}")

Total number of text chunks to embed: 24451


## Generate Embeddings

In [54]:
output = client.embeddings.create(input = combined_text,model=model_name)

APIConnectionError: Connection error.

In [ ]:
embeddings = output.data

In [ ]:
embeddings[0]

Embedding(embedding=[-0.0070558409206569195, 0.04594501107931137, -0.05385182797908783, -0.06533044576644897, 0.017553741112351418, 0.010379604063928127, 0.049455758184194565, 0.033520013093948364, -0.02570478431880474, -0.05702676624059677, 0.02364412695169449, -0.07534371316432953, 0.06471988558769226, -0.014363540336489677, 0.07583216577768326, 0.007983136922121048, -0.0360538586974144, 0.014607765711843967, 0.04789881780743599, -0.05073794722557068, -0.029826097190380096, -0.028742345049977303, -0.06044592708349228, -0.03654231131076813, 0.037519216537475586, 0.042525846511125565, 0.012989768758416176, -0.05760680139064789, -0.10245279222726822, -0.1552056074142456, 0.02916974015533924, 0.027215931564569473, 0.04728825390338898, 0.016744744032621384, 0.03028402104973793, -0.022987769916653633, -0.0046097650192677975, -0.01036433968693018, -0.024621030315756798, -0.025765839964151382, -0.03791608288884163, -0.043899618089199066, 0.03596227616071701, 0.01524885930120945, 0.0340389944

## Push data to database

### Initialize Pinecone and Create Index (This is Knowledge Base)

In [ ]:
# Wait for the index to be ready
while not pc.describe_index(index_name).status['ready']:
    time.sleep(1)

index = pc.Index(index_name)

vectors = []
for text, e in zip(texts, embeddings):
    entry_id = text.split(":")[0].strip()
    vectors.append({
        "id": entry_id,
        "values": e.embedding,
        "metadata": {'text': text}
    })
    
index.upsert(
    vectors=vectors,
    namespace="ns1"
)

## Get Closest documents

In [ ]:
output = client.embeddings.create(input = ["name 2 of students that an Artificial Intelligence engineer"],model=model_name)
embeding = output.data[0].embedding

In [ ]:
results = index.query(
    namespace="ns1",
    vector=embeding,
    top_k=3,
    include_values=False,
    include_metadata=True
)

print(results)